In [0]:
auto_loader_df = (spark.readStream # Initialise a Streaming Data Frame, doesn't just read once, instead it tracks changes.
    .format("cloudFiles") # Tell spark to use autoloader instead of standard file reader
    .option("cloudFiles.format", "json") # Tell autoloader to expect json format
    .option("cloudFiles.schemaLocation", "/Volumes/main/lufthansa/_checkpoints/bronze_flights/schema") # Autoloader will read a few file, guess the format for the schema and save it here. If the API changes later, it compares the new data to this saved schema (database)
    .option("multiLine", "true")  # Only put this if your json files are indented, otherwise if they are 1liners don't use this
    .load("/Volumes/main/lufthansa/landing_zone/ops/flights")) # Pointing to the source folder of files that you will be processing

query = (auto_loader_df.writeStream # Initliase the action phase; take instructions from the reader and aplpy them to the destination
    .format("delta") # Specify the output to be a delta table
    .option("checkpointLocation", "/Volumes/main/lufthansa/_checkpoints/bronze_flights/data") # This is the "Ledger", everytime a file is successfully processed it is saved here, so that when you run it again Spark will check here and ensure autoloader doesn't do unneccesary work
    .trigger(availableNow=True) # This is to have Batch like processing, so that your compute doesn't run forever
    .toTable("main.lufthansa.bronze_flights")) # This is the registration. Saves the data in the managed storage, give it a name in SQL format i.e. catalog.schema.table_name

query.awaitTermination() # Because Spark Streams run asynchronously, this tells notebook to not finish the cell until the stream has finished.

In [0]:
# RESET AUTOLOADERS CACHE

checkpoint_path = "/Volumes/main/lufthansa/_checkpoints/bronze_flights"
dbutils.fs.rm(checkpoint_path, recurse=True)

In [0]:
%sql
-- DESCRIBE DETAIL bronze;
-- SELECT * FROM bronze_flights
DROP TABLE main.lufthansa.bronze;

In [0]:
%sql
SELECT * FROM main.lufthansa.bronze_flights

In [0]:
%sql
-- This 'infers' the schema of the string on the fly just for viewing
SELECT schema_of_json(payload) FROM main.lufthansa.bronze_flights LIMIT 1;

In [0]:
%sql
-- This reaches inside the JSON string to grab a specific value
SELECT 
  payload:FlightStatusResource.Flights.Flight[0].OperatingCarrier.FlightNumber AS flight_no,
  payload:FlightStatusResource.Flights.Flight[0].Departure.ActualTimeLocal.DateTime AS departure_time
FROM main.lufthansa.bronze_flights